In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv = pd.read_csv(BASE_PATH + 'inventory_master.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')

# size of every table loaded
print('ledger', ledger.shape)
print('inv', inv.shape)
print('products', products.shape)
print('po_h', po_h.shape)
print('po_l', po_l.shape)
print('so_h', so_h.shape)
print('so_l', so_l.shape)

# columns and first rows of the ledger
print()
print(list(ledger.columns))
print(ledger.head())

# how many rows of each movement type
print()
print(ledger['movement_type'].value_counts())

Mounted at /content/drive
ledger (237230, 9)
inv (180, 8)
products (30, 23)
po_h (24000, 10)
po_l (155495, 9)
so_h (20000, 11)
so_l (130402, 9)

['movement_id', 'product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance']
      movement_id product_id branch_id movement_type movement_date  quantity  \
0  MOV-OUT-184742       P001    AHM001           OUT    2019-01-03         8   
1  MOV-OUT-208422       P001    AHM001           OUT    2019-01-05        12   
2   MOV-IN-111128       P001    AHM001            IN    2019-01-23       174   
3  MOV-OUT-159140       P001    AHM001           OUT    2019-01-25        12   
4  MOV-OUT-173693       P001    AHM001           OUT    2019-01-25        11   

  reference_type reference_id  running_balance  
0             SO    SO-157767              228  
1             SO    SO-168735              216  
2             PO    PO-181094              390  
3             SO    SO-947377       

In [2]:
# how big movements normally are, for each movement type on its own
size_summary = ledger.groupby('movement_type')['quantity'].describe()
print(size_summary)

# how many times bigger the largest movement is than the average
print()
print((ledger.groupby('movement_type')['quantity'].max() /
       ledger.groupby('movement_type')['quantity'].mean()).round(2))

                  count        mean        std   min   25%    50%    75%  \
movement_type                                                              
ADJUSTMENT       2454.0   27.298289  13.247121   5.0  16.0   27.0   38.0   
IN             127611.0  159.843548  81.141346  20.0  90.0  160.0  230.0   
OUT            107165.0   10.496263   5.776647   1.0   5.0   11.0   16.0   

                 max  
movement_type         
ADJUSTMENT      50.0  
IN             300.0  
OUT             20.0  

movement_type
ADJUSTMENT    1.83
IN            1.88
OUT           1.91
Name: quantity, dtype: float64


In [3]:
from scipy.stats import chisquare

# for each movement type: how evenly the rows spread over the quantity values
for t in ['IN', 'OUT', 'ADJUSTMENT']:
    q = ledger.loc[ledger['movement_type'] == t, 'quantity']
    counts = q.value_counts().sort_index()
    equal_share = 1 / len(counts)
    shares = counts / counts.sum()
    stat, p = chisquare(counts)
    print(t)
    print('  quantity values:', len(counts), '(from', counts.index.min(), 'to', counts.index.max(), ')')
    print('  equal share:', round(equal_share * 100, 3), 'percent')
    print('  smallest share:', round(shares.min() * 100, 3), 'percent at quantity', shares.idxmin())
    print('  largest share:', round(shares.max() * 100, 3), 'percent at quantity', shares.idxmax())
    print('  chi-square p-value:', round(p, 4))
    print()

IN
  quantity values: 281 (from 20 to 300 )
  equal share: 0.356 percent
  smallest share: 0.319 percent at quantity 226
  largest share: 0.397 percent at quantity 163
  chi-square p-value: 0.9377

OUT
  quantity values: 20 (from 1 to 20 )
  equal share: 5.0 percent
  smallest share: 4.881 percent at quantity 13
  largest share: 5.135 percent at quantity 12
  chi-square p-value: 0.2592

ADJUSTMENT
  quantity values: 46 (from 5 to 50 )
  equal share: 2.174 percent
  smallest share: 1.467 percent at quantity 21
  largest share: 2.689 percent at quantity 12
  chi-square p-value: 0.8916



In [4]:
# link every ledger row to the status of the order behind it
po_status_map = po_h.drop_duplicates('po_id').set_index('po_id')['po_status']
so_status_map = so_h.drop_duplicates('so_id').set_index('so_id')['order_status']

print('purchase orders in header:', len(po_h), ' distinct ids:', po_h['po_id'].nunique())
print('sales orders in header:', len(so_h), ' distinct ids:', so_h['so_id'].nunique())

led_status = ledger.copy()
led_status['order_status'] = led_status['reference_id'].map(po_status_map)
led_status['order_status'] = led_status['order_status'].fillna(led_status['reference_id'].map(so_status_map))

# adjustment rows have no order, so they are labelled ADJ
led_status.loc[led_status['reference_type'] == 'ADJ', 'order_status'] = 'ADJ'

# rows with no order status at all
print('rows with no order status:', led_status['order_status'].isna().sum())

# rows and units by movement type and order status
print()
print(led_status.groupby(['movement_type', 'order_status'])['quantity'].agg(rows='size', units='sum'))

purchase orders in header: 24000  distinct ids: 24000
sales orders in header: 20000  distinct ids: 20000
rows with no order status: 0

                              rows     units
movement_type order_status                  
ADJUSTMENT    ADJ             2454     66990
IN            Cancelled       1542    248556
              Received      126069  20149239
OUT           Cancelled       1275     13234
              Delivered     105890   1111598


In [5]:
# ledger rows that belong to cancelled orders
canc_rows = led_status[led_status['order_status'] == 'Cancelled']

canc_in = canc_rows[canc_rows['movement_type'] == 'IN']
canc_out = canc_rows[canc_rows['movement_type'] == 'OUT']

# how many cancelled orders exist and how many of them reach the ledger
print('cancelled purchase orders in header:', (po_h['po_status'] == 'Cancelled').sum())
print('  of these in the ledger:', canc_in['reference_id'].nunique())
print('cancelled sales orders in header:', (so_h['order_status'] == 'Cancelled').sum())
print('  of these in the ledger:', canc_out['reference_id'].nunique())

# how many ledger rows each cancelled order has
print()
print('ledger rows per cancelled purchase order:')
print(canc_in.groupby('reference_id').size().value_counts().sort_index())
print('ledger rows per cancelled sales order:')
print(canc_out.groupby('reference_id').size().value_counts().sort_index())

# dates of these rows
print()
print('first and last date, cancelled IN rows:', canc_in['movement_date'].min(), canc_in['movement_date'].max())
print('first and last date, cancelled OUT rows:', canc_out['movement_date'].min(), canc_out['movement_date'].max())

cancelled purchase orders in header: 2370
  of these in the ledger: 1098
cancelled sales orders in header: 1967
  of these in the ledger: 895

ledger rows per cancelled purchase order:
1    742
2    282
3     61
4     12
5      1
Name: count, dtype: int64
ledger rows per cancelled sales order:
1    591
2    238
3     58
4      6
5      2
Name: count, dtype: int64

first and last date, cancelled IN rows: 2019-01-14 2025-01-22
first and last date, cancelled OUT rows: 2019-01-05 2025-01-07


In [6]:
# which order ids appear in the ledger, for purchases (IN rows) and for sales (OUT rows)
ids_in_ledger_po = set(ledger.loc[ledger['movement_type'] == 'IN', 'reference_id'])
ids_in_ledger_so = set(ledger.loc[ledger['movement_type'] == 'OUT', 'reference_id'])

groups = [
    ('Received purchase orders', po_h.loc[po_h['po_status'] == 'Received', 'po_id'], ids_in_ledger_po),
    ('Cancelled purchase orders', po_h.loc[po_h['po_status'] == 'Cancelled', 'po_id'], ids_in_ledger_po),
    ('Delivered sales orders', so_h.loc[so_h['order_status'] == 'Delivered', 'so_id'], ids_in_ledger_so),
    ('Cancelled sales orders', so_h.loc[so_h['order_status'] == 'Cancelled', 'so_id'], ids_in_ledger_so),
]

# for each group: orders, orders found in the ledger, share
for name, ids, ledger_ids in groups:
    found = ids.isin(ledger_ids).sum()
    print(name)
    print('  orders:', len(ids))
    print('  in the ledger:', found)
    print('  share in the ledger:', round(found / len(ids) * 100, 2), 'percent')
    print()

Received purchase orders
  orders: 21630
  in the ledger: 21391
  share in the ledger: 98.9 percent

Cancelled purchase orders
  orders: 2370
  in the ledger: 1098
  share in the ledger: 46.33 percent

Delivered sales orders
  orders: 18033
  in the ledger: 17871
  share in the ledger: 99.1 percent

Cancelled sales orders
  orders: 1967
  in the ledger: 895
  share in the ledger: 45.5 percent



In [7]:
def compare_lines(lines, order_col, good_ids, ledger_part):
    # quantity on the order lines per order and product (repeated lines added together)
    wanted = (lines[lines[order_col].isin(good_ids)]
              .groupby([order_col, 'product_id'])['quantity'].sum()
              .rename('line_qty').reset_index())
    # quantity in the ledger per order and product
    got = (ledger_part.groupby(['reference_id', 'product_id'])['quantity'].sum()
           .rename('ledger_qty').reset_index()
           .rename(columns={'reference_id': order_col}))
    both = wanted.merge(got, on=[order_col, 'product_id'], how='outer', indicator=True)
    matched = both[both['_merge'] == 'both']
    print('order and product pairs on the order lines:', len(wanted))
    print('  ledger equals the order lines:', (matched['line_qty'] == matched['ledger_qty']).sum())
    print('  ledger lower than the order lines:', (matched['ledger_qty'] < matched['line_qty']).sum())
    print('  ledger higher than the order lines:', (matched['ledger_qty'] > matched['line_qty']).sum())
    print('  missing from the ledger:', (both['_merge'] == 'left_only').sum())
    print('  in the ledger with no order line:', (both['_merge'] == 'right_only').sum())
    return both

print('Received purchase orders')
po_compare = compare_lines(
    po_l, 'po_id',
    po_h.loc[po_h['po_status'] == 'Received', 'po_id'],
    led_status[(led_status['movement_type'] == 'IN') & (led_status['order_status'] == 'Received')])

print()
print('Delivered sales orders')
so_compare = compare_lines(
    so_l, 'so_id',
    so_h.loc[so_h['order_status'] == 'Delivered', 'so_id'],
    led_status[(led_status['movement_type'] == 'OUT') & (led_status['order_status'] == 'Delivered')])

Received purchase orders
order and product pairs on the order lines: 124417
  ledger equals the order lines: 110594
  ledger lower than the order lines: 2685
  ledger higher than the order lines: 0
  missing from the ledger: 11138
  in the ledger with no order line: 0

Delivered sales orders
order and product pairs on the order lines: 104418
  ledger equals the order lines: 92792
  ledger lower than the order lines: 2262
  ledger higher than the order lines: 0
  missing from the ledger: 9364
  in the ledger with no order line: 0


In [8]:
# purchase rows: does the ledger agree with the order header on date and branch
po_rows = led_status[(led_status['movement_type'] == 'IN') & (led_status['order_status'] == 'Received')]
po_rows = po_rows.merge(po_h[['po_id', 'branch_id', 'received_date']],
                        left_on='reference_id', right_on='po_id', how='left', suffixes=('', '_order'))
print('purchase rows:', len(po_rows))
print('  ledger date differs from received date:', (po_rows['movement_date'] != po_rows['received_date']).sum())
print('  ledger branch differs from order branch:', (po_rows['branch_id'] != po_rows['branch_id_order']).sum())

# sales rows: same question against the delivery date
so_rows = led_status[(led_status['movement_type'] == 'OUT') & (led_status['order_status'] == 'Delivered')]
so_rows = so_rows.merge(so_h[['so_id', 'branch_id', 'delivery_date']],
                        left_on='reference_id', right_on='so_id', how='left', suffixes=('', '_order'))
print()
print('sales rows:', len(so_rows))
print('  ledger date differs from delivery date:', (so_rows['movement_date'] != so_rows['delivery_date']).sum())
print('  ledger branch differs from order branch:', (so_rows['branch_id'] != so_rows['branch_id_order']).sum())

purchase rows: 126069
  ledger date differs from received date: 0
  ledger branch differs from order branch: 0

sales rows: 105890
  ledger date differs from delivery date: 0
  ledger branch differs from order branch: 0


In [9]:
# movement ids that appear more than once
print('ledger rows:', len(ledger))
print('distinct movement ids:', ledger['movement_id'].nunique())
print('rows whose movement id appears more than once:', ledger['movement_id'].duplicated(keep=False).sum())

# rows identical in everything except movement id and running balance
same_columns = ['product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id']
identical = ledger.duplicated(subset=same_columns, keep=False)
print()
print('rows identical to another row (apart from id and balance):', identical.sum())
print('groups of identical rows:', ledger[identical].groupby(same_columns).ngroups)
print()
print('by movement type:')
print(ledger[identical]['movement_type'].value_counts())

ledger rows: 237230
distinct movement ids: 237230
rows whose movement id appears more than once: 0

rows identical to another row (apart from id and balance): 1214
groups of identical rows: 606

by movement type:
movement_type
OUT    1114
IN      100
Name: count, dtype: int64


In [10]:
# order lines: how many lines repeat the same order, product and quantity
po_line_groups = (po_l.groupby(['po_id', 'product_id', 'quantity']).size()
                  .reset_index(name='lines').rename(columns={'po_id': 'reference_id'}))
so_line_groups = (so_l.groupby(['so_id', 'product_id', 'quantity']).size()
                  .reset_index(name='lines').rename(columns={'so_id': 'reference_id'}))
line_groups = pd.concat([po_line_groups, so_line_groups])

# the 606 groups of identical ledger rows, matched to the order lines
dup_groups = ledger[identical].groupby(same_columns).size().reset_index(name='ledger_rows')
matched_groups = dup_groups.merge(line_groups, on=['reference_id', 'product_id', 'quantity'], how='left')

print('groups of identical ledger rows:', len(matched_groups))
print('  order has two or more lines with that product and quantity:', (matched_groups['lines'] >= 2).sum())
print('  order has only one such line:', (matched_groups['lines'] == 1).sum())
print('  no matching order line found:', matched_groups['lines'].isna().sum())
print('  more ledger rows than order lines:', (matched_groups['ledger_rows'] > matched_groups['lines']).sum())

groups of identical ledger rows: 606
  order has two or more lines with that product and quantity: 606
  order has only one such line: 0
  no matching order line found: 0
  more ledger rows than order lines: 0


In [11]:
# order and product pairs where the ledger holds less than the order lines
lower_po = (po_compare[(po_compare['_merge'] == 'both') & (po_compare['ledger_qty'] < po_compare['line_qty'])]
            [['po_id', 'product_id']].rename(columns={'po_id': 'reference_id'}))
lower_so = (so_compare[(so_compare['_merge'] == 'both') & (so_compare['ledger_qty'] < so_compare['line_qty'])]
            [['so_id', 'product_id']].rename(columns={'so_id': 'reference_id'}))
lower_pairs = pd.concat([lower_po, lower_so])
lower_pairs['lower_than_order'] = True

# mark every ledger row
flagged = led_status.merge(lower_pairs, on=['reference_id', 'product_id'], how='left')
flagged['lower_than_order'] = flagged['lower_than_order'].notna()
flagged['cancelled_order'] = flagged['order_status'] == 'Cancelled'
flagged['flagged_any'] = flagged['lower_than_order'] | flagged['cancelled_order']

print('ledger rows after marking:', len(flagged))
print('order and product pairs marked lower:', len(lower_pairs))
print('rows tied to cancelled orders:', flagged['cancelled_order'].sum())
print('rows with a quantity lower than the order lines:', flagged['lower_than_order'].sum())
print('rows in both groups:', (flagged['cancelled_order'] & flagged['lower_than_order']).sum())
print('rows flagged in either group:', flagged['flagged_any'].sum())
print('share of the ledger flagged:', round(flagged['flagged_any'].mean() * 100, 2), 'percent')

print()
print('flagged rows by movement type:')
print(flagged[flagged['flagged_any']]['movement_type'].value_counts())

ledger rows after marking: 237230
order and product pairs marked lower: 4947
rows tied to cancelled orders: 2817
rows with a quantity lower than the order lines: 5521
rows in both groups: 0
rows flagged in either group: 8338
share of the ledger flagged: 3.51 percent

flagged rows by movement type:
movement_type
IN     4535
OUT    3803
Name: count, dtype: int64


In [12]:
# tables and columns that KPI 20 uses, with the quantity columns marked for zero and negative counts
used = [
    ('stock ledger', ledger, ['product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id'], ['quantity']),
    ('purchase orders header', po_h, ['po_id', 'branch_id', 'po_status', 'received_date'], []),
    ('sales orders header', so_h, ['so_id', 'branch_id', 'order_status', 'delivery_date'], []),
    ('purchase orders lines', po_l, ['po_id', 'product_id', 'quantity'], ['quantity']),
    ('sales orders lines', so_l, ['so_id', 'product_id', 'quantity'], ['quantity']),
]

for name, table, cols, number_cols in used:
    print(name, '-', len(table), 'rows')
    for c in cols:
        blanks = table[c].isna().sum() + (table[c].astype(str).str.strip() == '').sum()
        print('  blank', c, ':', blanks)
    for c in number_cols:
        print('  zero', c, ':', (table[c] == 0).sum(), ' negative', c, ':', (table[c] < 0).sum())
    print()

# received dates: blank only expected on cancelled purchase orders
print('blank received_date on Received purchase orders:', po_h.loc[po_h['po_status'] == 'Received', 'received_date'].isna().sum())
print('blank received_date on Cancelled purchase orders:', po_h.loc[po_h['po_status'] == 'Cancelled', 'received_date'].isna().sum())

stock ledger - 237230 rows
  blank product_id : 0
  blank branch_id : 0
  blank movement_type : 0
  blank movement_date : 0
  blank quantity : 0
  blank reference_type : 0
  blank reference_id : 0
  zero quantity : 0  negative quantity : 0

purchase orders header - 24000 rows
  blank po_id : 0
  blank branch_id : 0
  blank po_status : 0
  blank received_date : 2370

sales orders header - 20000 rows
  blank so_id : 0
  blank branch_id : 0
  blank order_status : 0
  blank delivery_date : 0

purchase orders lines - 155495 rows
  blank po_id : 0
  blank product_id : 0
  blank quantity : 0
  zero quantity : 0  negative quantity : 0

sales orders lines - 130402 rows
  blank so_id : 0
  blank product_id : 0
  blank quantity : 0
  zero quantity : 0  negative quantity : 0

blank received_date on Received purchase orders: 0
blank received_date on Cancelled purchase orders: 2370


In [13]:
def unit_summary(name, compare):
    both = compare[compare['_merge'] == 'both']
    lower = both[both['ledger_qty'] < both['line_qty']]
    line_units = int(compare['line_qty'].sum())
    ledger_units = int(compare['ledger_qty'].sum())
    short_lower = int((lower['line_qty'] - lower['ledger_qty']).sum())
    short_missing = int(compare.loc[compare['_merge'] == 'left_only', 'line_qty'].sum())
    print(name)
    print('  units on the order lines:', line_units)
    print('  units in the ledger:', ledger_units)
    print('  difference:', line_units - ledger_units)
    print('    from pairs where the ledger is lower:', short_lower)
    print('    from pairs missing from the ledger:', short_missing)
    print('    the two parts added up:', short_lower + short_missing)
    print()

unit_summary('Received purchase orders', po_compare)
unit_summary('Delivered sales orders', so_compare)

# units on the flagged ledger rows
print('units on rows tied to cancelled orders:', flagged.loc[flagged['cancelled_order'], 'quantity'].sum())
print('units on rows with a quantity lower than the order lines:', flagged.loc[flagged['lower_than_order'], 'quantity'].sum())

Received purchase orders
  units on the order lines: 22387123
  units in the ledger: 20149239
  difference: 2237884
    from pairs where the ledger is lower: 435404
    from pairs missing from the ledger: 1802480
    the two parts added up: 2237884

Delivered sales orders
  units on the order lines: 1234939
  units in the ledger: 1111598
  difference: 123341
    from pairs where the ledger is lower: 24216
    from pairs missing from the ledger: 99125
    the two parts added up: 123341

units on rows tied to cancelled orders: 261790
units on rows with a quantity lower than the order lines: 508520


In [14]:
# lookups for the order status and the quantities, built with plain loops
po_status_lookup = dict(zip(po_h['po_id'], po_h['po_status']))
so_status_lookup = dict(zip(so_h['so_id'], so_h['order_status']))

line_units = {}
for order, product, qty in zip(po_l['po_id'], po_l['product_id'], po_l['quantity']):
    line_units[(order, product)] = line_units.get((order, product), 0) + qty
for order, product, qty in zip(so_l['so_id'], so_l['product_id'], so_l['quantity']):
    line_units[(order, product)] = line_units.get((order, product), 0) + qty

ledger_units = {}
for order, product, qty in zip(ledger['reference_id'], ledger['product_id'], ledger['quantity']):
    ledger_units[(order, product)] = ledger_units.get((order, product), 0) + qty

# go through every ledger row and flag it
cancelled_ids = set()
lower_ids = set()
for move_id, order, product in zip(ledger['movement_id'], ledger['reference_id'], ledger['product_id']):
    status = po_status_lookup.get(order)
    if status is None:
        status = so_status_lookup.get(order)
    if status == 'Cancelled':
        cancelled_ids.add(move_id)
    elif status in ('Received', 'Delivered'):
        asked = line_units.get((order, product))
        if asked is not None and ledger_units[(order, product)] < asked:
            lower_ids.add(move_id)

second_flagged = cancelled_ids | lower_ids

# compare with the first method
first_flagged = set(flagged.loc[flagged['flagged_any'], 'movement_id'])
print('second method: cancelled rows', len(cancelled_ids), ' lower rows', len(lower_ids), ' flagged rows', len(second_flagged))
print('first method:  flagged rows', len(first_flagged))
print('flagged by the first method only:', len(first_flagged - second_flagged))
print('flagged by the second method only:', len(second_flagged - first_flagged))

second method: cancelled rows 2817  lower rows 5521  flagged rows 8338
first method:  flagged rows 8338
flagged by the first method only: 0
flagged by the second method only: 0


In [15]:
from scipy.stats import chi2_contingency

flagged['year'] = flagged['movement_date'].astype(str).str[:4]

for col in ['branch_id', 'year', 'product_id']:
    table = flagged.groupby(col)['flagged_any'].agg(rows='size', flagged='sum')
    table['share_percent'] = (table['flagged'] / table['rows'] * 100).round(2)
    print('flagged share by', col)
    if col == 'product_id':
        print(table.sort_values('share_percent').iloc[[0, -1]])
    else:
        print(table)
    chi2, p, dof, expected = chi2_contingency(pd.crosstab(flagged[col], flagged['flagged_any']))
    print('chi-square p-value:', round(p, 4))
    print()

flagged share by branch_id
            rows  flagged  share_percent
branch_id                               
AHM001     39911     1366           3.42
CHN001     44029     1482           3.37
DEL001     37967     1351           3.56
HYD001     36300     1329           3.66
KOL001     41272     1429           3.46
PUN001     37751     1381           3.66
chi-square p-value: 0.1089

flagged share by year
       rows  flagged  share_percent
year                               
2019  38373     1245           3.24
2020  40248     1466           3.64
2021  39714     1390           3.50
2022  39011     1381           3.54
2023  39773     1406           3.54
2024  38645     1412           3.65
2025   1466       38           2.59
chi-square p-value: 0.0126

flagged share by product_id
            rows  flagged  share_percent
product_id                              
P023        7873      247           3.14
P029        7904      332           4.20
chi-square p-value: 0.1897



In [16]:
year_sets = [
    ('all years', ['2019', '2020', '2021', '2022', '2023', '2024', '2025']),
    ('without 2025', ['2019', '2020', '2021', '2022', '2023', '2024']),
    ('2020 to 2024 only', ['2020', '2021', '2022', '2023', '2024']),
]

for label, years in year_sets:
    part = flagged[flagged['year'].isin(years)]
    shares = part.groupby('year')['flagged_any'].mean() * 100
    chi2, p, dof, expected = chi2_contingency(pd.crosstab(part['year'], part['flagged_any']))
    print(label)
    print('  lowest yearly share:', round(shares.min(), 2), 'percent in', shares.idxmin())
    print('  highest yearly share:', round(shares.max(), 2), 'percent in', shares.idxmax())
    print('  chi-square p-value:', round(p, 4))
    print()

all years
  lowest yearly share: 2.59 percent in 2025
  highest yearly share: 3.65 percent in 2024
  chi-square p-value: 0.0126

without 2025
  lowest yearly share: 3.24 percent in 2019
  highest yearly share: 3.65 percent in 2024
  chi-square p-value: 0.0285

2020 to 2024 only
  lowest yearly share: 3.5 percent in 2021
  highest yearly share: 3.65 percent in 2024
  chi-square p-value: 0.6997



In [17]:
flagged['month'] = flagged['movement_date'].astype(str).str[:7]
months = flagged.groupby('month')['flagged_any'].agg(rows='size', flagged='sum')
expected_months = set(pd.period_range('2019-01', '2025-01', freq='M').astype(str))

print('first and last ledger date:', flagged['movement_date'].min(), flagged['movement_date'].max())
print('months in the ledger:', len(months))
print('every month from 2019-01 to 2025-01 present:', set(months.index) == expected_months)
print('fewest rows in a month:', months['rows'].min(), 'in', months['rows'].idxmin())
print('most rows in a month:', months['rows'].max(), 'in', months['rows'].idxmax())
print('months with no flagged row:', (months['flagged'] == 0).sum())
print('fewest flagged rows in a month:', months['flagged'].min(), 'in', months['flagged'].idxmin())
print('most flagged rows in a month:', months['flagged'].max(), 'in', months['flagged'].idxmax())

flag_dates = flagged.loc[flagged['flagged_any'], 'movement_date']
print('first and last date of flagged rows:', flag_dates.min(), flag_dates.max())

first and last ledger date: 2019-01-01 2025-01-28
months in the ledger: 73
every month from 2019-01 to 2025-01 present: True
fewest rows in a month: 1466 in 2025-01
most rows in a month: 3846 in 2020-08
months with no flagged row: 0
fewest flagged rows in a month: 38 in 2025-01
most flagged rows in a month: 144 in 2020-08
first and last date of flagged rows: 2019-01-04 2025-01-28


In [18]:
import csv

def read_rows(file_name):
    with open(BASE_PATH + file_name, newline='') as f:
        return list(csv.DictReader(f))

raw_ledger = read_rows('stock_ledger.csv')
raw_po_head = read_rows('purchase_orders_header.csv')
raw_po_lines = read_rows('purchase_orders_lines.csv')
raw_so_head = read_rows('sales_orders_header.csv')
raw_so_lines = read_rows('sales_orders_lines.csv')

# one case of each kind, taken from the flag table (the first by movement id)
cancelled_case = flagged[flagged['cancelled_order'] & (flagged['movement_type'] == 'IN')].sort_values('movement_id').iloc[0]
lower_case = flagged[flagged['lower_than_order'] & (flagged['movement_type'] == 'OUT')].sort_values('movement_id').iloc[0]
ordinary_case = flagged[(~flagged['flagged_any']) & (flagged['movement_type'] == 'IN') & (flagged['order_status'] == 'Received')].sort_values('movement_id').iloc[0]

def rebuild_by_hand(label, row):
    order = row['reference_id']
    product = row['product_id']
    if order.startswith('PO'):
        head, lines, id_col, status_col = raw_po_head, raw_po_lines, 'po_id', 'po_status'
    else:
        head, lines, id_col, status_col = raw_so_head, raw_so_lines, 'so_id', 'order_status'

    ledger_rows = [r for r in raw_ledger if r['reference_id'] == order and r['product_id'] == product]
    ledger_units = sum(int(r['quantity']) for r in ledger_rows)
    status = next(r[status_col] for r in head if r[id_col] == order)
    line_units = sum(int(r['quantity']) for r in lines if r[id_col] == order and r['product_id'] == product)

    if status == 'Cancelled':
        by_hand = 'flagged: cancelled order'
    elif ledger_units < line_units:
        by_hand = 'flagged: ledger lower than the order lines'
    else:
        by_hand = 'not flagged'

    if row['cancelled_order']:
        in_table = 'flagged: cancelled order'
    elif row['lower_than_order']:
        in_table = 'flagged: ledger lower than the order lines'
    else:
        in_table = 'not flagged'

    print(label)
    print('  order and product:', order, product)
    print('  raw ledger rows:', len(ledger_rows), ' ledger units:', ledger_units)
    print('  order status:', status, ' units on the order lines:', line_units)
    print('  by hand:', by_hand)
    print('  in the table:', in_table)
    print('  they agree:', by_hand == in_table)
    print()

rebuild_by_hand('cancelled order row', cancelled_case)
rebuild_by_hand('lower quantity row', lower_case)
rebuild_by_hand('ordinary row', ordinary_case)

cancelled order row
  order and product: PO-171706 P028
  raw ledger rows: 1  ledger units: 89
  order status: Cancelled  units on the order lines: 89
  by hand: flagged: cancelled order
  in the table: flagged: cancelled order
  they agree: True

lower quantity row
  order and product: SO-508292 P008
  raw ledger rows: 1  ledger units: 14
  order status: Delivered  units on the order lines: 18
  by hand: flagged: ledger lower than the order lines
  in the table: flagged: ledger lower than the order lines
  they agree: True

ordinary row
  order and product: PO-141186 P004
  raw ledger rows: 1  ledger units: 234
  order status: Received  units on the order lines: 234
  by hand: not flagged
  in the table: not flagged
  they agree: True



In [19]:
# flagged share for purchase rows (IN) and sales rows (OUT)
in_out = flagged[flagged['movement_type'].isin(['IN', 'OUT'])]
by_type = in_out.groupby('movement_type')['flagged_any'].agg(rows='size', flagged='sum')
by_type['share_percent'] = (by_type['flagged'] / by_type['rows'] * 100).round(2)
print(by_type)
chi2, p, dof, expected = chi2_contingency(pd.crosstab(in_out['movement_type'], in_out['flagged_any']))
print('chi-square p-value:', round(p, 4))

# orders that contain at least one flagged row
print()
print('orders with a row tied to a cancelled order:', flagged.loc[flagged['cancelled_order'], 'reference_id'].nunique())
print('orders with a row lower than the order lines:', flagged.loc[flagged['lower_than_order'], 'reference_id'].nunique())
print('orders with any flagged row:', flagged.loc[flagged['flagged_any'], 'reference_id'].nunique())

                 rows  flagged  share_percent
movement_type                                
IN             127611     4535           3.55
OUT            107165     3803           3.55
chi-square p-value: 0.9566

orders with a row tied to a cancelled order: 1993
orders with a row lower than the order lines: 4602
orders with any flagged row: 6595
